# Section 1: Imports and Feature Definitions

In this section, we import the necessary scientific computing, machine learning, and visualization libraries. We also define the feature columns and target variable for the Random Forest model.

In [ ]:
from __future__ import annotations

import os
from typing import Dict, Any

import joblib
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

# Model features and target
FEATURES = ["movement_id", "scats_number", "dayofweek", "isweekend", "hour"]
TARGET = "hourly_traffic_volume"

# Section 2: Helper & Validation Functions

Here, we define standard helper functions to parse departure time formats and validate user inputs.

In [ ]:
def hhmm_to_hour(time_hhmm: str | int) -> int:
    """
    Convert HHMM input into hour.
    Example:
        1100 -> 11
        0830 -> 8
    """
    time_text = str(time_hhmm).strip().zfill(4)

    if len(time_text) != 4 or not time_text.isdigit():
        raise ValueError("Time must be in HHMM format, for example 1100 or 0830.")

    hour = int(time_text[:2])
    minute = int(time_text[2:])

    if hour < 0 or hour > 23:
        raise ValueError("Hour must be between 00 and 23.")

    if minute < 0 or minute > 59:
        raise ValueError("Minute must be between 00 and 59.")

    return hour


def validate_dayofweek(dayofweek: int) -> int:
    """Validate dayofweek where Monday=0 and Sunday=6."""
    dayofweek = int(dayofweek)

    if dayofweek < 0 or dayofweek > 6:
        raise ValueError("dayofweek must be 0 to 6. Monday=0, Sunday=6.")

    return dayofweek

# Section 3: Data Loading and Cleaning

This section implements the data loader. It cleans datatypes, parses datetime strings, and chronologically sorts records.

In [ ]:
def load_traffic_data(csv_path: str) -> pd.DataFrame:
    """
    Load and clean processed traffic_data.csv.
    """
    df = pd.read_csv(csv_path)

    required_columns = [
        "movement_id",
        "scats_number",
        "DateTime",
        "dayofweek",
        "isweekend",
        "hourly_traffic_volume",
    ]

    missing = [col for col in required_columns if col not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns in traffic data: {missing}")

    df["DateTime"] = pd.to_datetime(df["DateTime"], dayfirst=True, errors="coerce")
    df = df.dropna(subset=["DateTime"])

    df["movement_id"] = df["movement_id"].astype(str)
    df["scats_number"] = df["scats_number"].astype(int)
    df["dayofweek"] = df["dayofweek"].astype(int)
    df["isweekend"] = df["isweekend"].astype(int)
    df[TARGET] = df[TARGET].astype(float)

    # Extract hour from DateTime.
    df["hour"] = df["DateTime"].dt.hour.astype(int)

    # Keep records in time order to avoid data leakage during train/test split.
    df = df.sort_values(["DateTime", "movement_id"]).reset_index(drop=True)

    return df

# Section 4: Split Train/Test Set

In this section, we split the data chronologically (80% train, 20% test) to prevent time-series data leakage.

In [ ]:
def time_based_train_test_split(df: pd.DataFrame, train_ratio: float = 0.8):
    """
    Split data by DateTime chronologically.
    """
    if not 0 < train_ratio < 1:
        raise ValueError("train_ratio must be between 0 and 1.")

    unique_times = np.sort(df["DateTime"].unique())

    if len(unique_times) < 2:
        raise ValueError("Not enough unique DateTime values for train/test split.")

    cutoff = unique_times[int(len(unique_times) * train_ratio)]

    train_df = df[df["DateTime"] < cutoff].copy()
    test_df = df[df["DateTime"] >= cutoff].copy()

    return train_df, test_df, cutoff

# Section 5: Feature Engineering / Pipeline Setup

For Random Forest, we define a scikit-learn ColumnTransformer that encodes the categorical feature `movement_id` and passes numeric features directly.

In [ ]:
def build_random_forest_pipeline() -> Pipeline:
    """
    Create the preprocessing + Random Forest pipeline.
    """
    preprocessor = ColumnTransformer(
        transformers=[
            ("movement", OneHotEncoder(handle_unknown="ignore"), ["movement_id"]),
            ("numeric", "passthrough", ["scats_number", "dayofweek", "isweekend", "hour"]),
        ]
    )
    return preprocessor

# Section 6: Model Architecture & Compilation

Here, we initialize the model estimator (Random Forest Regressor) and bundle it with the preprocessor pipeline.

In [ ]:
def assemble_model_pipeline(preprocessor) -> Pipeline:
    rf_model = RandomForestRegressor(
        n_estimators=150,
        max_depth=18,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1,
    )

    pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", rf_model),
        ]
    )
    return pipeline

# Section 7: Model Training & Serialization

In this section, we train the model pipeline, save it as a joblib bundle, compute evaluation metrics, and write them to a metrics summary file.

In [ ]:
def evaluate_model(y_true, y_pred) -> Dict[str, float]:
    """Return regression evaluation metrics."""
    mae = mean_absolute_error(y_true, y_pred)
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    r2 = r2_score(y_true, y_pred)
    return {
        "MAE": float(mae),
        "RMSE": float(rmse),
        "R2": float(r2),
    }

def train_random_forest(
    data_path: str,
    output_path: str,
    metrics_output_path: str,
    train_ratio: float = 0.8,
) -> Dict[str, Any]:
    df = load_traffic_data(data_path)
    train_df, test_df, cutoff = time_based_train_test_split(df, train_ratio=train_ratio)
    
    preprocessor = build_random_forest_pipeline()
    pipeline = assemble_model_pipeline(preprocessor)

    print("Training Random Forest model...")
    print(f"Training rows: {len(train_df)}")
    print(f"Testing rows : {len(test_df)}")
    print(f"Cutoff time  : {pd.to_datetime(cutoff)}")

    pipeline.fit(train_df[FEATURES], train_df[TARGET])

    predictions = pipeline.predict(test_df[FEATURES])
    metrics = evaluate_model(test_df[TARGET], predictions)

    movement_to_scats = (
        df[["movement_id", "scats_number"]]
        .drop_duplicates()
        .set_index("movement_id")["scats_number"]
        .to_dict()
    )

    bundle = {
        "pipeline": pipeline,
        "features": FEATURES,
        "target": TARGET,
        "movement_to_scats": movement_to_scats,
        "metrics": metrics,
    }

    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    joblib.dump(bundle, output_path)

    # Save metrics in simple key-value report format.
    metrics_text = (
        f"Model : RandomForestRegressor\n"
        f"Target: {TARGET}\n"
        f"Features: {', '.join(FEATURES)}\n"
        f"Training Rows: {len(train_df)}\n"
        f"Testing Rows : {len(test_df)}\n"
        f"Cutoff Datetime: {pd.to_datetime(cutoff)}\n"
        f"MAE : {metrics['MAE']:.2f}\n"
        f"RMSE: {metrics['RMSE']:.2f}\n"
        f"R2  : {metrics['R2']:.4f}\n"
    )

    os.makedirs(os.path.dirname(metrics_output_path), exist_ok=True)
    with open(metrics_output_path, "w", encoding="utf-8") as file:
        file.write(metrics_text)

    print("\nRandom Forest Evaluation")
    print(f"MAE : {metrics['MAE']:.2f}")
    print(f"RMSE: {metrics['RMSE']:.2f}")
    print(f"R2  : {metrics['R2']:.4f}")
    print(f"\nSaved model   : {output_path}")
    print(f"Saved metrics : {metrics_output_path}")

    return bundle

# Section 8: Model Evaluation Visualization

Here, we visualize the model predictions against the actual test set target values using a scatter plot and sequential timeline comparison.

In [ ]:
DATA_PATH = "../data/processed/traffic_data.csv"
MODEL_OUT_PATH = "../models_saved/random_forest_model.pkl"
METRICS_OUT_PATH = "../models_saved/random_forest_metrics.csv"

bundle = train_random_forest(
    data_path=DATA_PATH,
    output_path=MODEL_OUT_PATH,
    metrics_output_path=METRICS_OUT_PATH
)

import matplotlib.pyplot as plt

# Load test data for visualization
df = load_traffic_data(DATA_PATH)
_, test_df, _ = time_based_train_test_split(df, train_ratio=0.8)

X_test = test_df[FEATURES]
y_actual = test_df[TARGET]
pipeline = bundle["pipeline"]
y_pred = pipeline.predict(X_test)

plt.figure(figsize=(15, 6))

# Plot 1: Scatter plot
plt.subplot(1, 2, 1)
plt.scatter(y_actual, y_pred, alpha=0.3, color="teal", edgecolors="none")
max_val = max(y_actual.max(), y_pred.max())
plt.plot([0, max_val], [0, max_val], color="darkorange", linestyle="--", lw=2, label="Perfect Alignment (y = x)")
plt.xlabel("Actual Traffic Volume (vehicles/hour)")
plt.ylabel("Predicted Traffic Volume (vehicles/hour)")
plt.title("Scatter Plot: Actual vs. Predicted Volume")
plt.legend()
plt.grid(True, linestyle=":", alpha=0.6)

# Plot 2: Timeline comparison
plt.subplot(1, 2, 2)
subset_size = min(150, len(test_df))
plt.plot(np.arange(subset_size), y_actual.iloc[:subset_size], label="Actual Volume", color="royalblue", alpha=0.8, lw=1.5)
plt.plot(np.arange(subset_size), y_pred[:subset_size], label="Predicted Volume", color="crimson", alpha=0.8, linestyle="--", lw=1.5)
plt.xlabel("Sample Index (Sequential)")
plt.ylabel("Traffic Volume (vehicles/hour)")
plt.title(f"Timeline Comparison (First {subset_size} samples)")
plt.legend()
plt.grid(True, linestyle=":", alpha=0.6)

plt.tight_layout()
plt.show()

# Section 9: Edge Predictions Batch Run

Finally, we load the road network topology and generate batch predictions for all network edges at a specific time and day (e.g. 11:00 AM on Monday).

In [ ]:
def predict_for_edges(
    model_path: str,
    edges_path: str,
    time_hhmm: str | int,
    dayofweek: int = 0,
    output_csv_path: str | None = None,
) -> pd.DataFrame:
    loaded = joblib.load(model_path)
    pipeline = loaded["pipeline"]
    movement_to_scats = loaded.get("movement_to_scats", {})

    edges_df = pd.read_csv(edges_path)

    required_edge_columns = ["from_site", "to_site", "movement_id"]
    missing = [col for col in required_edge_columns if col not in edges_df.columns]
    if missing:
        raise ValueError(f"Missing required columns in edges data: {missing}")

    hour = hhmm_to_hour(time_hhmm)
    dayofweek = validate_dayofweek(dayofweek)
    isweekend = 1 if dayofweek in [5, 6] else 0

    prediction_rows = []
    for _, edge in edges_df.iterrows():
        movement_id = str(edge["movement_id"])
        scats_number = movement_to_scats.get(movement_id, edge["from_site"])
        prediction_rows.append(
            {
                "movement_id": movement_id,
                "scats_number": int(scats_number),
                "dayofweek": int(dayofweek),
                "isweekend": int(isweekend),
                "hour": int(hour),
            }
        )

    X_pred = pd.DataFrame(prediction_rows)
    predicted_flow = pipeline.predict(X_pred)
    predicted_flow = np.maximum(0, predicted_flow)

    result = edges_df.copy()
    result["prediction_time_hhmm"] = str(time_hhmm).zfill(4)
    result["dayofweek"] = dayofweek
    result["predicted_hourly_traffic_volume"] = predicted_flow.round(2)

    if output_csv_path:
        os.makedirs(os.path.dirname(output_csv_path), exist_ok=True)
        result.to_csv(output_csv_path, index=False)
        print(f"Saved edge predictions: {output_csv_path}")

    return result

EDGES_PATH = "../data/processed/edges.csv"
PRED_OUT_PATH = "../data/processed/random_forest_edge_predictions.csv"

predictions_df = predict_for_edges(
    model_path=MODEL_OUT_PATH,
    edges_path=EDGES_PATH,
    time_hhmm="1100",
    dayofweek=0,
    output_csv_path=PRED_OUT_PATH
)

predictions_df.head()